In [1]:
pip install textdistance


  Using cached textdistance-4.6.3-py3-none-any.whl.metadata (18 kB)
Using cached textdistance-4.6.3-py3-none-any.whl (31 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import textdistance

# Load the normalized data
df = pd.read_csv("DATA SATYA/normalized_characters_all.csv")
df

,StoryID,SentenceID,Character,normalized_characters
0,1,0,Pan Karsa,pan karsa
1,1,0,pianakne muani,pianakne muani
2,1,8,Pan Karsa,pan karsa
3,1,10,pianakne,pianak
4,1,12,Pianakne,pianak
...,...,...,...,...
7402,120,106,I Yuyu,i yuyu
7403,120,106,kebone,kebo
7404,120,106,I Udang,i udang
7405,120,109,I Yuyu,i yuyu


In [3]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# Misalnya kamu sudah punya dataframe df
# Pastikan pilih kolom yang ingin dipakai (contoh: 'Character' atau 'normalized_characters')
text_data = " ".join(df['normalized_characters'].astype(str).tolist())

# Buat WordCloud
wc = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="viridis",   # bisa diganti misal "plasma", "inferno", "coolwarm"
    max_words=200
).generate(text_data)

# Tampilkan WordCloud
plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.show()

ModuleNotFoundError: No module named 'wordcloud'

RAYSSA

In [18]:
import re
import ast
import pandas as pd
import textdistance

def safe_parse(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        if x.startswith("[") and x.endswith("]"):
            try:
                return ast.literal_eval(x)
            except Exception:
                return [x]
        return [x]
    return []

# =========================
# Load data
# =========================
df = pd.read_csv("DATA SATYA/normalized_characters_all.csv")
df["normalized_characters"] = df["normalized_characters"].apply(safe_parse)

# --- Rapikan StoryID bila perlu (ambil angka pertama di string) ---
def normalize_story_id(v):
    if pd.isna(v):
        return None
    s = str(v)
    m = re.search(r"\d+", s)
    return int(m.group(0)) if m else s  # jika tak ada angka, pakai string as-is

if "StoryID" not in df.columns:
    raise KeyError("Kolom 'StoryID' tidak ditemukan pada CSV.")
df["StoryID"] = df["StoryID"].apply(normalize_story_id)

# =========================
# Util
# =========================
def normalize(s: str) -> str:
    return s.lower().strip()

def jaccard_sim(a: str, b: str) -> float:
    # Jaccard similarity (0..1, makin besar makin mirip)
    return float(textdistance.jaccard(a, b))

# =========================
# Clustering berbasis Jaccard
# =========================
def cluster_aliases(characters_list, jac_threshold=0.95):
    """
    - Tanpa pointer, tanpa exclude pairs, tanpa blacklist
    - Aturan:
      1) normalisasi lower/strip
      2) gabung jika substring (kedua arah)
      3) jika keduanya berakhiran sufiks umum ('ne' dsb.), bandingkan akar kata (tanpa sufiks) pakai Jaccard
      4) hindari false positive utk token <4: hanya gabung jika persis sama
      5) selain itu, pakai Jaccard >= threshold
    """
    clusters = {}   # key: Tokoh-i -> set(alias)
    next_id = 1
    suffixes = [("ne", 2)]  # tambahkan sufiks lain jika perlu, mis. ("nya", 3)

    for ch in characters_list:
        ch = normalize(ch)
        placed = False

        for key, cluster in clusters.items():
            for name in list(cluster):
                name_n = normalize(name)

                # 1) Substring match dua arah
                if f" {ch} " in f" {name_n} " or f" {name_n} " in f" {ch} ":
                    cluster.add(ch); placed = True; break

                # 2) Bandingkan akar untuk sufiks umum
                for suf, slen in suffixes:
                    if ch.endswith(suf) and name_n.endswith(suf):
                        if jaccard_sim(ch[:-slen], name_n[:-slen]) >= jac_threshold:
                            cluster.add(ch); placed = True
                        break
                if placed:
                    break

                # 3) Token pendek: (<4) hanya gabung bila persis
                if len(ch) < 4 or len(name_n) < 4:
                    if ch == name_n:
                        cluster.add(ch); placed = True
                    if placed: break
                    continue

                # 4) Similarity Jaccard umum
                if jaccard_sim(ch, name_n) >= jac_threshold:
                    cluster.add(ch); placed = True; break

            if placed:
                break

        if not placed:
            clusters[f"Tokoh-{next_id}"] = {ch}
            next_id += 1

    # set -> list, urutkan agar rapi (panjang desc)
    return {k: sorted(list(v), key=len, reverse=True) for k, v in clusters.items()}

# =========================
# Proses per StoryID (tanpa hubungan antar story)
# =========================
THRESH = 0.95  # catatan: Jaccard cenderung ketat; turunkan jika terlalu pecah (mis. 0.85)
all_results = []

# buang baris tanpa storyid atau tanpa karakter
df = df[df["StoryID"].notna()]
df = df[df["normalized_characters"].apply(lambda x: isinstance(x, list) and len(x) > 0)]

# Flatten list per StoryID
grouped = df.groupby("StoryID")["normalized_characters"].apply(lambda x: sum(x, []))

for story_id, characters in grouped.items():
    uniq_chars = sorted(set(characters), key=len, reverse=True)
    result = cluster_aliases(uniq_chars, jac_threshold=THRESH)

    for person, aliases in result.items():
        all_results.append({
            "StoryID": story_id,
            "CharactersID": person,
            "AliasCharacters": aliases
        })

# =========================
# Save
# =========================
df_result = pd.DataFrame(all_results)
outfile = f"alias_jaccard_{int(THRESH*100)}.csv"  # contoh: alias_jaccard_95.csv
df_result.to_csv(outfile, index=False)
print(f"✅ Alias clustering selesai (tanpa pointer/exclude/blacklist), per-story only, Jaccard≥{THRESH:.2f}. Disimpan ke {outfile}")


✅ Alias clustering selesai (tanpa pointer/exclude/blacklist), per-story only, Jaccard≥0.95. Disimpan ke alias_jaccard_95.csv


In [19]:
import pandas as pd
import ast
import re
from itertools import combinations
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# =========================
# KONFIGURASI FILE
# =========================
GROUND_TRUTH_CSV = "DATA SATYA/string_similarity_bali.csv"  # GT
PREDICT_CSV      = "alias_jaccard_95.csv"           # Prediksi

# Nama kolom standar (otomatis dinormalkan kalau beda casing)
COL_STORY   = "StoryID"
COL_CLUSTER = "CharactersID"
COL_ALIASES = "AliasCharacters"

# =========================
# PARSER & NORMALIZER
# =========================
_punct_tail = re.compile(r"[,\.;:!\?\)]*$")  # hapus tanda baca di ekor token

def norm_alias(s: str) -> str:
    return _punct_tail.sub("", str(s).strip().lower())

def parse_aliases_cell(x):
    """Dukung:
       - list Python (['a','b'])
       - stringified list ("['a','b']")
       - string dipisah koma/semicolon ("a, b; c")
       - string tunggal ("a")"""
    if isinstance(x, list):
        return [norm_alias(t) for t in x if str(t).strip()]
    if isinstance(x, str):
        s = x.strip()
        if s.startswith('[') and s.endswith(']'):
            try:
                lst = ast.literal_eval(s)
                return [norm_alias(t) for t in lst if str(t).strip()]
            except Exception:
                pass
        # split by comma/semicolon jika ada
        if ("," in s) or (";" in s):
            parts = re.split(r"[;,]", s)
            return [norm_alias(t) for t in parts if str(t).strip()]
        # satu alias
        return [norm_alias(s)] if s else []
    return []

def read_clusters(csv_path):
    """Kembalikan dict: {StoryID: {ClusterID: [alias,…]}}"""
    df = pd.read_csv(csv_path)

    # selaraskan nama kolom jika beda penulisan
    rename_map = {}
    for c in df.columns:
        lc = c.strip().lower()
        if lc == "storyid":      rename_map[c] = COL_STORY
        if lc in ("charactersid","person","cluster","tokoh"): rename_map[c] = COL_CLUSTER
        if lc in ("aliascharacters","aliases","alias","listalias","characters"): rename_map[c] = COL_ALIASES
    if rename_map:
        df = df.rename(columns=rename_map)

    for must in [COL_STORY, COL_CLUSTER, COL_ALIASES]:
        if must not in df.columns:
            raise KeyError(f"Kolom wajib '{must}' tidak ada di {csv_path}. Kolom: {list(df.columns)}")

    df[COL_ALIASES] = df[COL_ALIASES].apply(parse_aliases_cell)

    clusters_per_story = {}
    for sid, sub in df.groupby(COL_STORY):
        story = {}
        for _, r in sub.iterrows():
            cid = str(r[COL_CLUSTER])
            aliases = r[COL_ALIASES]
            if not aliases: 
                continue
            story.setdefault(cid, set()).update(aliases)
        # set → list & sort
        story = {k: sorted(v) for k, v in story.items() if v}
        if story:
            clusters_per_story[sid] = story
    return clusters_per_story

# =========================
# BACA DATA → gt, pr
# =========================
gt = read_clusters(GROUND_TRUTH_CSV)
pr = read_clusters(PREDICT_CSV)

# =========================
# EVALUASI (pairwise + ARI/NMI)
# =========================
def to_label_map(cluster_dict):
    m = {}
    for i, (cid, aliases) in enumerate(sorted(cluster_dict.items(), key=lambda x: str(x[0]))):
        for a in aliases:
            m[a] = i
    return m

def pairwise_confusion(gt_map, pr_map):
    aliases = sorted(set(gt_map) & set(pr_map))
    if len(aliases) < 2:
        return 0,0,0,0,0
    TP=FP=FN=TN=0; total=0
    for i in range(len(aliases)):
        for j in range(i+1, len(aliases)):
            a, b = aliases[i], aliases[j]
            sg = (gt_map[a]==gt_map[b])
            sp = (pr_map[a]==pr_map[b])
            total += 1
            if  sg and  sp: TP += 1
            elif not sg and sp: FP += 1
            elif sg and not sp: FN += 1
            else: TN += 1
    return TP,FP,FN,TN,total

def safe_div(a,b): return a/b if b else 0.0
def metrics(TP,FP,FN,TN):
    precision = safe_div(TP, TP+FP)
    recall    = safe_div(TP, TP+FN)
    f1        = safe_div(2*precision*recall, precision+recall)
    accuracy  = safe_div(TP+TN, TP+FP+FN+TN)
    return accuracy, precision, recall, f1

# Per-story, macro, micro, ARI/NMI
rows=[]
micro_TP=micro_FP=micro_FN=micro_TN=0
y_true=[]; y_pred=[]

common_sids = sorted(set(gt) & set(pr))
if not common_sids:
    raise ValueError("Tidak ada StoryID yang sama di GT dan Prediksi.")

for sid in common_sids:
    gt_map = to_label_map(gt[sid])
    pr_map = to_label_map(pr[sid])

    TP,FP,FN,TN,total = pairwise_confusion(gt_map, pr_map)
    if total==0:
        continue

    acc, prec, rec, f1 = metrics(TP,FP,FN,TN)
    rows.append({
        "StoryID": sid,
        "pairs": total,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })

    micro_TP += TP; micro_FP += FP; micro_FN += FN; micro_TN += TN

    common_aliases = sorted(set(gt_map) & set(pr_map))
    y_true.extend([gt_map[a] for a in common_aliases])
    y_pred.extend([pr_map[a] for a in common_aliases])

# Per-story report
df_report = pd.DataFrame(rows).sort_values(["StoryID","pairs"], ascending=[True,False]).reset_index(drop=True)
print("=== Per-Story Report (head) ===")
print(df_report.head(10))

# Macro average
macro_acc  = df_report["Accuracy"].mean()  if not df_report.empty else 0.0
macro_prec = df_report["Precision"].mean() if not df_report.empty else 0.0
macro_rec  = df_report["Recall"].mean()    if not df_report.empty else 0.0
macro_f1   = df_report["F1"].mean()        if not df_report.empty else 0.0

# Micro average (overall)
overall_acc, overall_prec, overall_rec, overall_f1 = metrics(micro_TP, micro_FP, micro_FN, micro_TN)

# ARI & NMI
overall_ari = adjusted_rand_score(y_true, y_pred) if y_true else 0.0
overall_nmi = normalized_mutual_info_score(y_true, y_pred) if y_true else 0.0

print("\n=== Macro Average (per-story mean) ===")
print(f"Accuracy : {macro_acc:.4f}")
print(f"Precision: {macro_prec:.4f}")
print(f"Recall   : {macro_rec:.4f}")
print(f"F1       : {macro_f1:.4f}")

print("\n=== Overall (Micro) ===")
print(f"Accuracy : {overall_acc:.4f}")
print(f"Precision: {overall_prec:.4f}")
print(f"Recall   : {overall_rec:.4f}")
print(f"F1       : {overall_f1:.4f}")

print("\n=== Information-Theoretic ===")
print(f"ARI      : {overall_ari:.4f}")
print(f"NMI      : {overall_nmi:.4f}")

=== Per-Story Report (head) ===
   StoryID  pairs  Accuracy  Precision    Recall        F1
0        1     15  0.866667        0.0  0.000000  0.000000
1        2     28  0.892857        1.0  0.625000  0.769231
2        3    120  0.941667        0.5  0.142857  0.222222
3        4     78  0.846154        1.0  0.250000  0.400000
4        5     15  0.866667        1.0  0.500000  0.666667
5        6     66  0.924242        1.0  0.375000  0.545455
6        7     36  0.888889        0.0  0.000000  0.000000
7        8    153  0.915033        1.0  0.187500  0.315789
8        9     78  0.756410        1.0  0.095238  0.173913
9       10    153  0.973856        1.0  0.555556  0.714286

=== Macro Average (per-story mean) ===
Accuracy : 0.8869
Precision: 0.7272
Recall   : 0.3399
F1       : 0.4259

=== Overall (Micro) ===
Accuracy : 0.9208
Precision: 0.7657
Recall   : 0.2567
F1       : 0.3845

=== Information-Theoretic ===
ARI      : 0.0171
NMI      : 0.1140


BALINLP

In [4]:
import balinese_nlp

In [55]:
import re
import ast
import pandas as pd

# -------------------------------------------------
# Import model alias clustering (rule-based)
# -------------------------------------------------
# Pastikan paket sudah terpasang:
# pip install balinese-nlp==2.0.2
from balinese_nlp.narratives.aliasclustering.rule_based import AliasClusteringRuleBased

# =========================
# Helper
# =========================
def safe_parse(x):
    """Ubah string '["a","b"]' -> list; string biasa -> [string]; kosong -> []"""
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        if x.startswith("[") and x.endswith("]"):
            try:
                return ast.literal_eval(x)
            except Exception:
                return [x]
        return [x]
    return []


def normalize_story_id(v):
    """Ambil angka pertama di StoryID (jika ada), selain itu pakai string apa adanya."""
    if pd.isna(v):
        return None
    s = str(v)
    m = re.search(r"\d+", s)
    return int(m.group(0)) if m else s


def normalize_cluster_output(results):
    """
    Samakan bentuk keluaran cluster() menjadi dict: {cluster_id: [alias, ...]}.
    Mengantisipasi beberapa kemungkinan format dari library.
    """
    # Case 1: sudah dict {label: list/iterable}
    if isinstance(results, dict):
        norm = {}
        for k, members in results.items():
            # pastikan list unik & rapi (urut panjang desc lalu alfabet)
            uniq = sorted(set(members), key=lambda s: (-len(str(s)), str(s)))
            norm[str(k)] = [str(x) for x in uniq]
        return norm

    # Case 2: list of iterables: [ {"members":[...]}, ... ] atau [ [...], ... ]
    if isinstance(results, list):
        norm = {}
        for i, cluster in enumerate(results, start=1):
            if isinstance(cluster, dict) and "members" in cluster:
                members = cluster["members"]
            else:
                members = cluster
            uniq = sorted({str(x) for x in members}, key=lambda s: (-len(s), s))
            norm[f"Tokoh-{i}"] = uniq
        return norm

    # Fallback: bungkus semua jadi satu cluster
    return {"Tokoh-1": [str(results)]}


# =========================
# Load data
# =========================
df = pd.read_csv("DATA SATYA/normalized_characters_all.csv")

if "normalized_characters" not in df.columns:
    raise KeyError("Kolom 'normalized_characters' tidak ditemukan pada CSV.")

df["normalized_characters"] = df["normalized_characters"].apply(safe_parse)

if "StoryID" not in df.columns:
    raise KeyError("Kolom 'StoryID' tidak ditemukan pada CSV.")
df["StoryID"] = df["StoryID"].apply(normalize_story_id)


# =========================
# Filter baris valid
# =========================
df = df[df["StoryID"].notna()]
df = df[df["normalized_characters"].apply(lambda x: isinstance(x, list) and len(x) > 0)]

# Gabungkan daftar karakter per StoryID (flatten)
grouped = df.groupby("StoryID")["normalized_characters"].apply(lambda x: sum(x, []))


# =========================
# Proses per StoryID dengan AliasClusteringRuleBased
# =========================
PAIRWISE = "jaccard"
THRESH = 0.8

all_results = []

for story_id, characters in grouped.items():
    # Unik-kan dan rapikan (opsional): urutkan dari panjang ke pendek agar stabil
    list_of_identified_characters = sorted(set(map(str, characters)), key=lambda s: (-len(s), s))

    # --- create object from our class
    aliasclustering_model = AliasClusteringRuleBased(
        pairwise_distance=PAIRWISE,
        avg_threshold_similarity=THRESH,
    )

    # --- fit a list of identified characters
    aliasclustering_model.fit(list_of_identified_characters)

    # --- run cluster() method
    results = aliasclustering_model.cluster()

    # --- Normalisasi bentuk hasil agar konsisten {cluster_id: [aliases]}
    results_norm = normalize_cluster_output(results)

    # --- Kumpulkan ke list untuk DataFrame keluaran
    for cluster_id, aliases in results_norm.items():
        all_results.append(
            {
                "StoryID": story_id,
                "CharactersID": str(cluster_id),
                "AliasCharacters": aliases,  # biarkan sebagai list; Pandas akan simpan sebagai string list
            }
        )

# =========================
# Save
# =========================
df_result = pd.DataFrame(all_results)

outfile = f"alias_rulebased_{PAIRWISE.replace('-','')}_{int(THRESH*100)}.csv"
df_result.to_csv(outfile, index=False)

print(
    f"✅ Alias clustering selesai dengan AliasClusteringRuleBased "
    f"({PAIRWISE}, ≥{THRESH:.2f}) per-story. Disimpan ke {outfile}"
)


✅ Alias clustering selesai dengan AliasClusteringRuleBased (jaccard, ≥0.80) per-story. Disimpan ke alias_rulebased_jaccard_80.csv


In [56]:
import pandas as pd
import ast
import re
from itertools import combinations
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# =========================
# KONFIGURASI FILE
# =========================
GROUND_TRUTH_CSV = "DATA SATYA/string_similarity_bali.csv"  # GT
PREDICT_CSV      = "alias_rulebased_jaccard_80.csv"           # Prediksi

# Nama kolom standar (otomatis dinormalkan kalau beda casing)
COL_STORY   = "StoryID"
COL_CLUSTER = "CharactersID"
COL_ALIASES = "AliasCharacters"

# =========================
# PARSER & NORMALIZER
# =========================
_punct_tail = re.compile(r"[,\.;:!\?\)]*$")  # hapus tanda baca di ekor token

def norm_alias(s: str) -> str:
    return _punct_tail.sub("", str(s).strip().lower())

def parse_aliases_cell(x):
    """Dukung:
       - list Python (['a','b'])
       - stringified list ("['a','b']")
       - string dipisah koma/semicolon ("a, b; c")
       - string tunggal ("a")"""
    if isinstance(x, list):
        return [norm_alias(t) for t in x if str(t).strip()]
    if isinstance(x, str):
        s = x.strip()
        if s.startswith('[') and s.endswith(']'):
            try:
                lst = ast.literal_eval(s)
                return [norm_alias(t) for t in lst if str(t).strip()]
            except Exception:
                pass
        # split by comma/semicolon jika ada
        if ("," in s) or (";" in s):
            parts = re.split(r"[;,]", s)
            return [norm_alias(t) for t in parts if str(t).strip()]
        # satu alias
        return [norm_alias(s)] if s else []
    return []

def read_clusters(csv_path):
    """Kembalikan dict: {StoryID: {ClusterID: [alias,…]}}"""
    df = pd.read_csv(csv_path)

    # selaraskan nama kolom jika beda penulisan
    rename_map = {}
    for c in df.columns:
        lc = c.strip().lower()
        if lc == "storyid":      rename_map[c] = COL_STORY
        if lc in ("charactersid","person","cluster","tokoh"): rename_map[c] = COL_CLUSTER
        if lc in ("aliascharacters","aliases","alias","listalias","characters"): rename_map[c] = COL_ALIASES
    if rename_map:
        df = df.rename(columns=rename_map)

    for must in [COL_STORY, COL_CLUSTER, COL_ALIASES]:
        if must not in df.columns:
            raise KeyError(f"Kolom wajib '{must}' tidak ada di {csv_path}. Kolom: {list(df.columns)}")

    df[COL_ALIASES] = df[COL_ALIASES].apply(parse_aliases_cell)

    clusters_per_story = {}
    for sid, sub in df.groupby(COL_STORY):
        story = {}
        for _, r in sub.iterrows():
            cid = str(r[COL_CLUSTER])
            aliases = r[COL_ALIASES]
            if not aliases: 
                continue
            story.setdefault(cid, set()).update(aliases)
        # set → list & sort
        story = {k: sorted(v) for k, v in story.items() if v}
        if story:
            clusters_per_story[sid] = story
    return clusters_per_story

# =========================
# BACA DATA → gt, pr
# =========================
gt = read_clusters(GROUND_TRUTH_CSV)
pr = read_clusters(PREDICT_CSV)

# =========================
# EVALUASI (pairwise + ARI/NMI)
# =========================
def to_label_map(cluster_dict):
    m = {}
    for i, (cid, aliases) in enumerate(sorted(cluster_dict.items(), key=lambda x: str(x[0]))):
        for a in aliases:
            m[a] = i
    return m

def pairwise_confusion(gt_map, pr_map):
    aliases = sorted(set(gt_map) & set(pr_map))
    if len(aliases) < 2:
        return 0,0,0,0,0
    TP=FP=FN=TN=0; total=0
    for i in range(len(aliases)):
        for j in range(i+1, len(aliases)):
            a, b = aliases[i], aliases[j]
            sg = (gt_map[a]==gt_map[b])
            sp = (pr_map[a]==pr_map[b])
            total += 1
            if  sg and  sp: TP += 1
            elif not sg and sp: FP += 1
            elif sg and not sp: FN += 1
            else: TN += 1
    return TP,FP,FN,TN,total

def safe_div(a,b): return a/b if b else 0.0
def metrics(TP,FP,FN,TN):
    precision = safe_div(TP, TP+FP)
    recall    = safe_div(TP, TP+FN)
    f1        = safe_div(2*precision*recall, precision+recall)
    accuracy  = safe_div(TP+TN, TP+FP+FN+TN)
    return accuracy, precision, recall, f1

# Per-story, macro, micro, ARI/NMI
rows=[]
micro_TP=micro_FP=micro_FN=micro_TN=0
y_true=[]; y_pred=[]

common_sids = sorted(set(gt) & set(pr))
if not common_sids:
    raise ValueError("Tidak ada StoryID yang sama di GT dan Prediksi.")

for sid in common_sids:
    gt_map = to_label_map(gt[sid])
    pr_map = to_label_map(pr[sid])

    TP,FP,FN,TN,total = pairwise_confusion(gt_map, pr_map)
    if total==0:
        continue

    acc, prec, rec, f1 = metrics(TP,FP,FN,TN)
    rows.append({
        "StoryID": sid,
        "pairs": total,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })

    micro_TP += TP; micro_FP += FP; micro_FN += FN; micro_TN += TN

    common_aliases = sorted(set(gt_map) & set(pr_map))
    y_true.extend([gt_map[a] for a in common_aliases])
    y_pred.extend([pr_map[a] for a in common_aliases])

# Per-story report
df_report = pd.DataFrame(rows).sort_values(["StoryID","pairs"], ascending=[True,False]).reset_index(drop=True)
print("=== Per-Story Report (head) ===")
print(df_report.head(10))

# Macro average
macro_acc  = df_report["Accuracy"].mean()  if not df_report.empty else 0.0
macro_prec = df_report["Precision"].mean() if not df_report.empty else 0.0
macro_rec  = df_report["Recall"].mean()    if not df_report.empty else 0.0
macro_f1   = df_report["F1"].mean()        if not df_report.empty else 0.0

# Micro average (overall)
overall_acc, overall_prec, overall_rec, overall_f1 = metrics(micro_TP, micro_FP, micro_FN, micro_TN)

# ARI & NMI
overall_ari = adjusted_rand_score(y_true, y_pred) if y_true else 0.0
overall_nmi = normalized_mutual_info_score(y_true, y_pred) if y_true else 0.0

print("\n=== Macro Average (per-story mean) ===")
print(f"Accuracy : {macro_acc:.4f}")
print(f"Precision: {macro_prec:.4f}")
print(f"Recall   : {macro_rec:.4f}")
print(f"F1       : {macro_f1:.4f}")

print("\n=== Overall (Micro) ===")
print(f"Accuracy : {overall_acc:.4f}")
print(f"Precision: {overall_prec:.4f}")
print(f"Recall   : {overall_rec:.4f}")
print(f"F1       : {overall_f1:.4f}")

print("\n=== Information-Theoretic ===")
print(f"ARI      : {overall_ari:.4f}")
print(f"NMI      : {overall_nmi:.4f}")

=== Per-Story Report (head) ===
   StoryID  pairs  Accuracy  Precision    Recall        F1
0        1     15  0.866667      0.000  0.000000  0.000000
1        2     28  0.857143      1.000  0.500000  0.666667
2        3    120  0.941667      0.500  0.285714  0.363636
3        4     78  0.846154      0.625  0.625000  0.625000
4        5     15  0.866667      1.000  0.500000  0.666667
5        6     66  0.969697      1.000  0.750000  0.857143
6        7     55  0.890909      0.400  0.400000  0.400000
7        8    153  0.928105      1.000  0.312500  0.476190
8        9     78  0.820513      1.000  0.333333  0.500000
9       10    153  0.980392      1.000  0.666667  0.800000

=== Macro Average (per-story mean) ===
Accuracy : 0.9205
Precision: 0.8021
Recall   : 0.6123
F1       : 0.6583

=== Overall (Micro) ===
Accuracy : 0.9398
Precision: 0.7435
Recall   : 0.5717
F1       : 0.6464

=== Information-Theoretic ===
ARI      : 0.0254
NMI      : 0.1186
